# CALCUL DE LA MATRICE DE RECOUVREMENT D'HAGEDORN

Ce script calcule numériquement la matrice de recouvrement **S** entre deux bases de fonctions d'Hagedorn avec une haute précision.

## Fonctions de base d'Hagedorn

Les fonctions de base sont définies par :


\begin{aligned}
b_i(q, \alpha_1, q_1, p_1) &= N_1 \cdot H_{i-1}\left(\sqrt{\Re(\alpha_1)}(q-q_1)\right) \cdot \exp\left(-\frac{i}{2}\alpha_1(q-q_1)^2 + ip_1(q-q_1)\right) \\
b_j(q, \alpha_2, q_2, p_2) &= N_2 \cdot H_{j-1}\left(\sqrt{\Re(\alpha_2)}(q-q_2)\right) \cdot \exp\left(-\frac{i}{2}\alpha_2(q-q_2)^2 + ip_2(q-q_2)\right)
\end{aligned}


où :
- $ H_n $ sont les polynômes d'Hermite
- $ N_1, N_2 $ sont des constantes de normalisation
-  $\alpha$ paramètres de largeur (complexes avec partie réelle positive)
- $q_1, p_1, q_2, p_2 $ paramètres de centre et impulsion

## Matrice de recouvrement

L'élément $ S_{ij} $ de la matrice est donné par :


\begin{equation}
S_{ij} = \int_{-\infty}^{+\infty} b_i^*(q) \cdot b_j(q) \, dq
\end{equation}

**Auteur :** Script généré pour le calcul numérique précis

In [18]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CALCUL DE MATRICES DE RECOUVREMENT ENTRE BASES D'HAGEDORN

Ce script calcule la matrice de recouvrement S[i,j] entre deux bases d'Hagedorn:

b_i(q) = N_i * H_i(√Re(α₁) * (q-q₁)) * exp(-α₁/2 * (q-q₁)² + ip₁*(q-q₁))
b_j(q) = N_j * H_j(√Re(α₂) * (q-q₂)) * exp(-α₂/2 * (q-q₂)² + ip₂*(q-q₂))

où S[i,j] = ∫ b_i*(q) * b_j(q) dq

Auteur: Script généré pour le calcul numérique haute précision
Version: 1.0
"""

import numpy as np
from scipy.special import hermite
from scipy.integrate import quad
import warnings
import math as math
warnings.filterwarnings('ignore')

def hagedorn_basis_function(q, n, alpha, q0, p0):
    """
    Calcule une fonction de base d'Hagedorn.

    Paramètres:
    -----------
    q : float ou array
        Variable de position
    n : int
        Indice de la fonction (ordre du polynôme d'Hermite)
    alpha : complex
        Paramètre complexe alpha de la base
    q0 : float
        Position du centre du paquet d'onde
    p0 : float
        Impulsion du centre du paquet d'onde

    Retourne:
    ---------
    complex
        Valeur de la fonction de base d'Hagedorn

    Notes:
    ------
    La fonction est définie comme:
    b_n(q) = N_n * H_n(√Re(α) * (q-q₀)) * exp(-α/2 * (q-q₀)² + ip₀*(q-q₀))

    où N_n = (Re(α)/π)^(1/4) / √(2ⁿ * n!) est le facteur de normalisation
    """
    re_alpha = np.real(alpha)
    im_alpha = np.imag(alpha)

    # Facteur de normalisation
    N = (re_alpha / np.pi)**(1/4) / np.sqrt(2**n * math.factorial(n))

    # Variable réduite pour le polynôme d'Hermite
    xi = np.sqrt(re_alpha) * (q - q0)

    # Polynôme d'Hermite d'ordre n
    H_poly = hermite(n)
    H_val = H_poly(xi)

    # Phase complexe complète
    gaussian_part = -re_alpha/2 * (q - q0)**2
    momentum_part = 1j * p0 * (q - q0)
    complex_part = 1j * im_alpha/2 * (q - q0)**2

    phase = gaussian_part + momentum_part + complex_part

    return N * H_val * np.exp(phase)

def calculate_overlap_matrix(n_max, alpha1, q1, p1, alpha2, q2, p2):
    """
    Calcule la matrice de recouvrement entre deux bases d'Hagedorn.

    Paramètres:
    -----------
    n_max : int
        Dimension maximale de la base (indices de 0 à n_max-1)
    alpha1, q1, p1 : complex, float, float
        Paramètres de la première base
    alpha2, q2, p2 : complex, float, float
        Paramètres de la deuxième base

    Retourne:
    ---------
    ndarray (n_max, n_max) complex
        Matrice de recouvrement S où S[i,j] = ⟨b_i|b_j⟩

    Notes:
    ------
    L'intégration est effectuée avec une haute précision numérique.
    Les limites d'intégration sont automatiquement adaptées aux paramètres.
    """
    print(f"Calcul de la matrice de recouvrement {n_max}×{n_max}")
    print(f"Base 1: α₁={alpha1}, q₁={q1}, p₁={p1}")
    print(f"Base 2: α₂={alpha2}, q₂={q2}, p₂={p2}")

    S = np.zeros((n_max, n_max), dtype=complex)

    # Calcul adaptatif des limites d'intégration
    re_alpha1, re_alpha2 = np.real(alpha1), np.real(alpha2)
    sigma1 = 1.0 / np.sqrt(re_alpha1)  # Largeur caractéristique base 1
    sigma2 = 1.0 / np.sqrt(re_alpha2)  # Largeur caractéristique base 2

    q_center = (q1 + q2) / 2
    q_range = 6 * max(sigma1, sigma2) + abs(q1 - q2) + 2
    q_min = q_center - q_range
    q_max = q_center + q_range

    print(f"Limites d'intégration: [{q_min:.2f}, {q_max:.2f}]")

    def integrand_real(q, i, j):
        """Partie réelle de l'intégrande ⟨b_i|b_j⟩"""
        psi_i = hagedorn_basis_function(q, i, alpha1, q1, p1)
        psi_j = hagedorn_basis_function(q, j, alpha2, q2, p2)
        return np.real(np.conj(psi_i) * psi_j)

    def integrand_imag(q, i, j):
        """Partie imaginaire de l'intégrande ⟨b_i|b_j⟩"""
        psi_i = hagedorn_basis_function(q, i, alpha1, q1, p1)
        psi_j = hagedorn_basis_function(q, j, alpha2, q2, p2)
        return np.imag(np.conj(psi_i) * psi_j)

    # Calcul de chaque élément de matrice
    for i in range(n_max):
        for j in range(n_max):
            # Intégration haute précision
            real_part, _ = quad(integrand_real, q_min, q_max, args=(i, j),
                              epsabs=1e-12, epsrel=1e-12, limit=150)
            imag_part, _ = quad(integrand_imag, q_min, q_max, args=(i, j),
                              epsabs=1e-12, epsrel=1e-12, limit=150)

            S[i, j] = real_part + 1j * imag_part

            # Nettoyage numérique des très petites parties imaginaires
            if np.abs(S[i, j].imag) < 1e-14:
                S[i, j] = S[i, j].real + 0j

    print("✓ Calcul terminé avec succès")
    return S

def print_matrix_detailed(S, title):
    """
    Affiche la matrice ligne par ligne avec formatage détaillé.

    Paramètres:
    -----------
    S : ndarray complex
        Matrice à afficher
    title : str
        Titre de la matrice
    """
    print(f"\n{title}")
    print("=" * max(50, len(title)))

    n, m = S.shape
    for i in range(n):
        elements = []
        for j in range(m):
            val = S[i, j]
            if np.abs(val.imag) < 1e-12:
                # Nombre réel
                elements.append(f"{val.real:8.6f}     ")
            else:
                # Nombre complexe
                if val.imag >= 0:
                    elements.append(f"{val.real:6.4f}+{val.imag:6.4f}j")
                else:
                    elements.append(f"{val.real:6.4f}{val.imag:6.4f}j")

        print(f"Ligne {i}: [" + " ".join(elements) + "]")

def test_orthonormality(alpha, q0, p0, n_max=3):
    """
    Teste l'orthonormalité d'une base d'Hagedorn.

    Paramètres:
    -----------
    alpha, q0, p0 : paramètres de la base
    n_max : int, nombre de fonctions à tester
    """
    print(f"\nTest d'orthonormalité pour α={alpha}, q₀={q0}, p₀={p0}")
    print("-" * 50)

    def test_overlap(n, m):
        def integrand(q):
            psi_n = hagedorn_basis_function(q, n, alpha, q0, p0)
            psi_m = hagedorn_basis_function(q, m, alpha, q0, p0)
            return np.real(np.conj(psi_n) * psi_m)

        result, _ = quad(integrand, -10, 10, epsabs=1e-12)
        return result

    for n in range(n_max):
        for m in range(n, n_max):
            overlap = test_overlap(n, m)
            expected = 1.0 if n == m else 0.0
            print(f"⟨φ_{n}|φ_{m}⟩ = {overlap:.8f} (attendu: {expected:.1f})")

# ============================================================================
# PROGRAMME PRINCIPAL - TROIS CAS DE TEST
# ============================================================================

def main():

    n_max = 3  # Dimension des matrices

    alpha1 = 1.0 + 0.1j
    q1 = 0.5
    p1 = 0.25
    
    alpha2  = 1.0 + 0.2j
    q2 = 0.49
    p2 = 0.24

    S1 = calculate_overlap_matrix(n_max, alpha1, q1, p1, alpha2, q2, p2)
    print_matrix_detailed(S1, "S₁")

if __name__ == "__main__":
    main()


Calcul de la matrice de recouvrement 3×3
Base 1: α₁=(1+0.1j), q₁=0.5, p₁=0.25
Base 2: α₂=(1+0.2j), q₂=0.49, p₂=0.24
Limites d'intégration: [-7.51, 8.50]
✓ Calcul terminé avec succès

S₁
Ligne 0: [0.9990+0.0274j 0.0075-0.0058j -0.0027+0.0351j]
Ligne 1: [-0.0066-0.0062j 0.9950+0.0772j 0.0113-0.0079j]
Ligne 2: [-0.0027+0.0352j -0.0087-0.0089j 0.9874+0.1264j]


In [ ]:
bonjour ;
voici deu fonction de deux base différentes d'Hagedorn:
\begin{aligned}
b_i(q, \alpha_1, q_1, p_1) &= N_1 \cdot H_{i-1}\left(\sqrt{\Re(\alpha_1)}(q-q_1)
\right) \cdot \exp\left(-\frac{i}{2}\alpha_1(q-q_1)^2 + ip_1(q-q_1)\right) \\
b_j(q, \alpha_2, q_2, p_2) &= N_2 \cdot H_{j-1}\left(\sqrt{\Re(\alpha_2)}(q-q_2)
\right) \cdot \exp\left(-\frac{i}{2}\alpha_2(q-q_2)^2 + ip_2(q-q_2)\right)
\end{aligned}

je veux calculer la matrice de recouvrement avec la quadrature de gauss.
en utilisant une grille intermédiaire.
la quadrature de gausse est définie par :
call hercom(nq, x(:), w(:)), ou x et w sont les point de grille te w les poids.
trouve une grille (q construit en utilisant qe,sqe qui sont obtenu a partir de q1,q2,p1,p2,alpha1,alpha1).

SyntaxError: invalid syntax (3907194595.py, line 1)